In [2]:
!git clone https://github.com/Coding-With-Josh/langnostic.git
%cd langnostic
!pip -q install transformers datasets peft jiwer soundfile librosa accelerate
import torch; print(torch.cuda.is_available())

Cloning into 'langnostic'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 74 (delta 28), reused 62 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 85.93 KiB | 1.26 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/langnostic
True


In [4]:
!pip install -q jiwer peft

In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


True Tesla T4


In [3]:
import io, json, itertools, os, sys, time
import soundfile as sf
from datasets import load_dataset, Audio
from transformers import pipeline

sys.path.insert(0, ".")
from speechgap.report import load_jsonl, score, to_markdown

N = 100
GROUPS = [("yoruba_read", "yo_ng", "yoruba"), ("english_read", "en_us", "english")]

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
               device=0, chunk_length_s=30)
print("device", asr.model.device, flush=True)

os.makedirs("results/cache", exist_ok=True)
path = "results/cache/fleurs_zero_shot.jsonl"
t0 = time.time()

with open(path, "w", encoding="utf-8") as f:
    for group, config, language in GROUPS:
        ds = load_dataset("google/fleurs", config, split="test", streaming=True)
        ds = ds.cast_column("audio", Audio(decode=False))
        for i, row in enumerate(itertools.islice(ds, N)):
            if i == 0:
                print(group, "first row", round(time.time() - t0), "s", flush=True)
            wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            hyp = asr({"raw": wav.astype("float32"), "sampling_rate": sr},
                      generate_kwargs={"language": language, "task": "transcribe"})["text"]
            rec = {"utt_id": f"{config}_{i}", "speaker": f"{config}_{i}", "group": group,
                   "ref": row["raw_transcription"], "hyp": hyp}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            if i % 10 == 0:
                print(group, i, round(time.time() - t0), "s", flush=True)

recs = load_jsonl(path)
print(to_markdown(score(recs)))
for r in [r for r in recs if r["group"] == "yoruba_read"][:3]:
    print("ref:", r["ref"])
    print("hyp:", r["hyp"], "\n")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

... (12 outputs left out)

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


yoruba_read 0 32 s


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


yoruba_read 10 101 s
yoruba_read 20 182 s
yoruba_read 30 289 s
yoruba_read 40 350 s
yoruba_read 50 422 s
yoruba_read 60 492 s
yoruba_read 70 583 s
yoruba_read 80 673 s
yoruba_read 90 719 s
english_read first row 796 s
english_read 0 797 s
english_read 10 805 s
english_read 20 814 s
english_read 30 823 s
english_read 40 830 s
english_read 50 839 s
english_read 60 848 s
english_read 70 856 s
english_read 80 863 s
english_read 90 872 s
| group | clips | speakers | wer [95% ci] | cer [95% ci] | insertions share | halluc. |
| --- | --- | --- | --- | --- | --- | --- |
| english_read | 100 | 100 | 6.5% [5.1%, 7.9%] | 3.0% [2.3%, 3.8%] | 17% | 0% |
| yoruba_read | 100 | 100 | 367.4% [307.9%, 427.9%] | 145.6% [123.9%, 170.9%] | 73% | 49% |
ref: Àwọn èyàn ti mọ̀ nípa àwọn kemika pepe bí wúrà, fàdákà àti kọ́pa àtijọ́, torípé a lè rí wọn ní ìṣẹ̀dá ní ipò àdáyéba wọn, kò dẹ̀ le jù láti wú wọn jáda láti inú ilẹ̀ pẹ̀lú irinṣẹ́ àtilẹ̀bá.
hyp:  ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ 

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!cp results/cache/fleurs_zero_shot.jsonl /content/drive/MyDrive/

Mounted at /content/drive


In [6]:
import io, itertools, time
import soundfile as sf
from datasets import load_dataset, Audio
from transformers import pipeline
from speechgap.report import score, to_markdown

ds = load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))
rows = []
for row in itertools.islice(ds, 20):
    wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    rows.append((wav.astype("float32"), sr, row["raw_transcription"]))

asr_nochunk = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0)

variants = {
    "default": (asr, {}),
    "guard": (asr, {"no_repeat_ngram_size": 4, "max_new_tokens": 256}),
    "guard_nochunk": (asr_nochunk, {"no_repeat_ngram_size": 4, "max_new_tokens": 256}),
}

recs = []
for name, (pipe, extra) in variants.items():
    t = time.time()
    for i, (wav, sr, ref) in enumerate(rows):
        hyp = pipe({"raw": wav, "sampling_rate": sr},
                   generate_kwargs={"language": "yoruba", "task": "transcribe", **extra})["text"]
        recs.append({"utt_id": f"{name}_{i}", "speaker": f"{name}_{i}", "group": name,
                     "ref": ref, "hyp": hyp})
    print(name, round(time.time() - t), "s", flush=True)

print(to_markdown(score(recs)))
for name in variants:
    r = next(r for r in recs if r["group"] == name)
    print(name, "|", r["hyp"][:150])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

default 155 s


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

guard 52 s


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

guard_nochunk 53 s
| group | clips | speakers | wer [95% ci] | cer [95% ci] | insertions share | halluc. |
| --- | --- | --- | --- | --- | --- | --- |
| default | 20 | 20 | 327.5% [217.7%, 458.2%] | 159.4% [101.8%, 250.7%] | 70% | 50% |
| guard | 20 | 20 | 127.1% [107.2%, 150.6%] | 75.4% [64.8%, 85.9%] | 24% | 10% |
| guard_nochunk | 20 | 20 | 127.1% [107.2%, 150.6%] | 75.4% [64.8%, 85.9%] | 24% | 10% |
default |  ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ ḍ
guard |  ន្់់ ន ្ក្ ធក ប្, វក ្ ក់, ល្. ន, ប, ធ, ក, ភ, ត, ផ, ឡ, ស, ង, ហ, ឣ, ន. វ, ម, យ, ឝ, រ, ឦ, ឍ, ឧ, ឬ, ឰ, ឯ, ឤ, ញ, ឩ, ឮ, ឱ, ឭ, ឫ, ី, ូ, ឺ, ិ, ឴, ឪ, ា, ឨ, ើ
guard_nochunk |  ន្់់ ន ្ក្ ធក ប្, វក ្ ក់, ល្. ន, ប, ធ, ក, ភ, ត, ផ, ឡ, ស, ង, ហ, ឣ, ន. វ, ម, យ, ឝ, រ, ឦ, ឍ, ឧ, ឬ, ឰ, ឯ, ឤ, ញ, ឩ, ឮ, ឱ, ឭ, ឫ, ី, ូ, ឺ, ិ, ឴, ឪ, ា, ឨ, ើ


In [7]:
feats = asr.feature_extractor(rows[0][0], sampling_rate=rows[0][1],
                              return_tensors="pt").input_features.to("cuda")
out = asr.model.generate(feats, language="yoruba", task="transcribe",
                         max_new_tokens=20, no_repeat_ngram_size=4)
print(asr.tokenizer.decode(out[0], skip_special_tokens=False))

[transformers] Both `max_new_tokens` (=20) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 ḍᵃᵃ ᵃᶦᵃ �


In [8]:
import json
with open("fleurs_yo_sensitivity.jsonl", "w", encoding="utf-8") as f:
    for r in recs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
from google.colab import files
files.download("fleurs_yo_sensitivity.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>